# Stock Forecasting (refactored)
Run this notebook from the extracted project directory **or** its `ml/` subdirectory. The first code cell searches upwards for the package root and adds it to Python's module path. This notebook demonstrates the tested project modules and saves artifacts to the project's `outputs/` directory. Requires internet for live market data. Not investment advice.

In [3]:
import os
import json
import sys
from pathlib import Path

# Jupyter may start in the ml/ folder. Discover the extracted project root
# so `from ml.run_training import run` works from either directory.
CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (CWD, *CWD.parents)
     if (folder / "ml" / "run_training.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate ml/run_training.py. Extract the entire ZIP and start "
        "Jupyter from Stock-Forecasting-Naive-ML-Dashboard/ or its ml/ folder."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.graph_objects as go
from dotenv import load_dotenv
from ml.run_training import run
from ml.src.forecast import artifact_slug

load_dotenv(PROJECT_ROOT / ".env")
TICKER = os.getenv("STOCK", "AAPL")
PERIOD = os.getenv("PERIOD", "5y")
INTERVAL = os.getenv("INTERVAL", "1d")
EXCHANGE = os.getenv("EXCHANGE", "NYSE")
print("Project root:", PROJECT_ROOT)


Project root: C:\Users\kaush\Files\Codes\Projects\Stock Forecasting


## Train + validate + evaluate
The CLI pipeline uses training-only scalers, horizon-aware evaluation and a naïve baseline.

In [4]:
metrics, forecast = run(TICKER, period=PERIOD, interval=INTERVAL, exchange=EXCHANGE, epochs=30, lookback=30, horizon=7, output_dir=PROJECT_ROOT / "outputs")
print(json.dumps(metrics, indent=2))

C:\Users\kaush\Files\Codes\Projects\Stock Forecasting\ml\src\transformer.py:45: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


Epoch 001/30 | train=0.085971 | val=0.033022
Epoch 002/30 | train=0.025954 | val=0.030899
Epoch 003/30 | train=0.020237 | val=0.015099
Epoch 004/30 | train=0.016837 | val=0.015612
Epoch 005/30 | train=0.014990 | val=0.012436
Epoch 006/30 | train=0.013321 | val=0.012234
Epoch 007/30 | train=0.012232 | val=0.009223
Epoch 008/30 | train=0.010903 | val=0.008629
Epoch 009/30 | train=0.009901 | val=0.007646
Epoch 010/30 | train=0.008619 | val=0.006008
Epoch 011/30 | train=0.007921 | val=0.006203
Epoch 012/30 | train=0.007127 | val=0.006315
Epoch 013/30 | train=0.006478 | val=0.005736
Epoch 014/30 | train=0.005558 | val=0.006239
Epoch 015/30 | train=0.005249 | val=0.005744
Epoch 016/30 | train=0.004837 | val=0.005457
Epoch 017/30 | train=0.004328 | val=0.005576
Epoch 018/30 | train=0.004094 | val=0.004563
Epoch 019/30 | train=0.003895 | val=0.004721
Epoch 020/30 | train=0.003587 | val=0.006051
Epoch 021/30 | train=0.003529 | val=0.005462
Epoch 022/30 | train=0.003252 | val=0.005231
Epoch 023/

## Future exchange sessions

In [5]:
display(forecast)

,Forecast_Date,Predicted_Close
0,2026-11-23,328.541260
1,2026-11-24,346.346039
2,2026-11-25,310.045013
3,2026-11-27,238.822845
4,2026-11-30,281.771942
5,2026-12-01,291.441620
6,2026-12-02,309.899200


## Per-horizon error and baseline

In [6]:
days = list(metrics["per_horizon"])
fig = go.Figure()
fig.add_trace(go.Bar(x=days, y=[metrics["per_horizon"][d]["MAE"] for d in days], name="Transformer"))
fig.add_trace(go.Bar(x=days, y=[metrics["naive_baseline"]["per_horizon"][d]["MAE"] for d in days], name="Naïve last close"))
fig.update_layout(barmode="group", title="Holdout mean absolute error by forecast horizon", xaxis_title="Forecast horizon", yaxis_title="MAE in price units")
fig.show()

## Training loss (Huber, not MSE)

In [7]:
history = json.loads((PROJECT_ROOT / "outputs" / f"{artifact_slug(TICKER)}_history.json").read_text())
fig = go.Figure()
fig.add_trace(go.Scatter(y=history["train_loss"], name="Train"))
fig.add_trace(go.Scatter(y=history["val_loss"], name="Validation"))
fig.update_layout(title="Huber loss during training", xaxis_title="Epoch", yaxis_title="Huber loss")
fig.show()